In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from category_encoders import BinaryEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.ensemble import RandomForestRegressor

from sklearn.feature_selection import (
    mutual_info_classif,
    chi2,
    f_classif,
    SelectKBest
)
from sklearn.metrics import roc_auc_score, classification_report
import re

In [23]:
df_real_estate = pd.read_csv('Data/Master_RealEstate_Combined_v1.csv')
df = df_real_estate.copy()
df.sample(5)

,Type,Address,Price,Price_Per_M2,Area,Rooms,Bathrooms,Floor,Finish,View,Amenities,Amenities_Count,Is_Installments
6926,Building,Shorouk,3375000.0,14673.913043,230.0,NaN,NaN,First Floor,1,Garden,"Elevator, Balcony, Electricity Meter, Natural Gas",4,1
7781,Commercial,Madinaty,8500000.0,61151.079137,139.0,3.0,3.0,Not Mentioned,1,Other,NaN,0,0
3868,Apartment,Madinaty,9000000.0,57324.840764,157.0,3.0,3.0,Not Mentioned,1,Garden,Garden,1,0
7541,Medical,Madinaty,4250000.0,NaN,NaN,2.0,1.0,Not Mentioned,1,Garden,NaN,0,0
4963,Building,Other,5000000.0,28571.428571,175.0,3.0,2.0,First Floor,1,Other,"Elevator, Electricity Meter, Natural Gas",3,0


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8865 entries, 0 to 8864
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Type             8865 non-null   str    
 1   Address          8865 non-null   str    
 2   Price            8864 non-null   float64
 3   Price_Per_M2     5846 non-null   float64
 4   Area             5792 non-null   float64
 5   Rooms            6919 non-null   float64
 6   Bathrooms        7018 non-null   float64
 7   Floor            8276 non-null   str    
 8   Finish           8865 non-null   int64  
 9   View             8839 non-null   str    
 10  Amenities        5405 non-null   str    
 11  Amenities_Count  8865 non-null   int64  
 12  Is_Installments  8865 non-null   int64  
dtypes: float64(5), int64(3), str(5)
memory usage: 1.4 MB


In [25]:
df.describe()

,Price,Price_Per_M2,Area,Rooms,Bathrooms,Finish,Amenities_Count,Is_Installments
count,8.864000e+03,5.846000e+03,5792.000000,6.919000e+03,7018.000000,8865.000000,8865.000000,8865.000000
mean,8.931347e+06,inf,235.296961,1.453384e+08,2.744514,0.950818,1.372589,0.434856
std,1.303270e+07,NaN,1468.082564,5.404888e+09,7.079778,0.216260,1.597618,0.495766
min,3.500000e+03,2.000000e+01,0.000000,1.000000e+00,1.000000,0.000000,0.000000,0.000000
25%,4.340598e+06,2.709975e+04,120.000000,2.000000e+00,2.000000,1.000000,0.000000,0.000000
50%,6.621527e+06,4.357100e+04,157.500000,3.000000e+00,2.000000,1.000000,1.000000,0.000000
75%,1.020000e+07,6.403981e+04,200.000000,3.000000e+00,3.000000,1.000000,2.000000,1.000000
max,5.999490e+08,inf,63000.000000,2.011171e+11,180.000000,1.000000,7.000000,1.000000


In [26]:
# Cap price and area at 99th percentile
for col in ["Price", "Area"]:
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=upper)

In [27]:
df["View"].value_counts()

View
Garden         3523
Other          1854
Main Street    1442
Pool            763
Side Street     345
Club            267
Corner          178
Lake            167
Sea View        117
Back             53
Plaza            43
Nile View        40
Golf Course      28
Seaview          19
Name: count, dtype: int64

### Amenities column

In [28]:
df["Amenities"].value_counts()

Amenities
Garden                                                                    1642
Security                                                                   220
Elevator                                                                   199
Pool                                                                       174
Parking                                                                    151
                                                                          ... 
Security, Pool, Parking, Electricity Meter                                   1
Pool, Electricity Meter                                                      1
Elevator, Garden, Parking, Electricity Meter, Water Meter, Natural Gas       1
Security, Electricity Meter, Water Meter, Natural Gas                        1
Elevator, Security, Electricity Meter, Water Meter, Natural Gas              1
Name: count, Length: 284, dtype: int64

In [29]:
amenity_groups = {
    "has_utilities": ["Electricity Meter", "Water Meter", "Natural Gas"],
    "has_security": ["Security"],
    "has_luxury": ["Pool", "Garden", "Solar"],
    "has_access": ["Elevator", "Parking"],
}


def check_group(amenity_string, keywords):
    if pd.isna(amenity_string):
        return 0
    return int(any(kw in amenity_string for kw in keywords))


for group, keywords in amenity_groups.items():
    df[group] = df["Amenities"].apply(lambda x: check_group(x, keywords))

In [30]:
df = df.drop(["Amenities"], axis=1)

### Address column

In [31]:
df["Address"].value_counts()

Address
New Cairo                                                 2172
Madinaty                                                  1141
Other                                                      940
New Capital                                                314
Rehab                                                      305
                                                          ... 
ashgaar city                                                 1
alexandria smouha compounds in smouha fayrouzit smouha       1
sheraton heliopolis                                          1
cairo new cairo compounds jrdyny hyts 3                      1
cairo el shorouk compounds wesal city                        1
Name: count, Length: 1338, dtype: int64

In [32]:
noise_words = [
    "city",
    "compound",
    "compounds",
    "hy",
    "mdyn",
    "kmbwnd",
    "lstd",
    "jdyl",
    "el",
    "al",
    "of",
    "the",
    "kwrnysh",
    "lnyl",
]
def clean_address(x):
    x = str(x).lower()  # lowercase
    x = re.sub(r"[^a-z\s]", " ", x)  # remove numbers & symbols
    x = re.sub(r"\s+", " ", x).strip()  # remove extra spaces
    words = x.split()
    words = [w for w in words if w not in noise_words]
    return " ".join(words[:4])

df["Address"] = df["Address"].apply(clean_address)

df["Address"].value_counts()

Address
new cairo                 2172
madinaty                  1141
other                      940
new capital                314
rehab                      305
                          ... 
cairo new cairo palma        1
ashgaar                      1
sheraton heliopolis          1
cairo new cairo jrdyny       1
cairo shorouk wesal          1
Name: count, Length: 691, dtype: int64

In [33]:
def extract_city(x):
    x = str(x).lower()
    x = re.sub(r"[^a-z\s]", " ", x)

    if (
        "new capital" in x
        or "capital" in x
        or "eins park" in x
        or "mall" in x
        or "inizio" in x
        or "administrative" in x
        or "track rev" in x
    ):
        return "new capital"

    elif "madinaty" in x:
        return "madinaty"

    elif "sheikh zayed" in x:
        return "sheikh zayed"

    elif "maadi" in x:
        return "maadi"

    elif "rehab" in x:
        return "rehab"

    elif "shorouk" in x:
        return "shorouk"

    elif "heliopolis" in x:
        return "heliopolis"

    elif (
        "october" in x
        or "6th of october" in x
        or "th october" in x
        or "kite residence" in x
        or "isola" in x
        or "ashgaar" in x
        or "advida" in x
        or "lark" in x
        or "calm residence" in x
        or "v levels" in x
        or "belong" in x
    ):
        return "october"

    elif "nasr city" in x or "nasr" in x:
        return "nasr city"

    elif "alexandria" in x:
        return "alexandria"

    elif "north coast" in x:
        return "north coast"

    elif (
        "red sea" in x
        or "hurghada" in x
        or "ain elsokhna" in x
        or "groove" in x
        or "baymount" in x
        or "panorama hills resort" in x
    ):
        return "hurghada"

    elif "damietta" in x:
        return "damietta"

    elif "gharbia" in x:
        return "gharbia"

    elif "dakahlia" in x:
        return "dakahlia"

    elif "port said" in x:
        return "port said"

    elif "ras sidr" in x:
        return "ras sidr"

    elif "marsa matruh" in x:
        return "marsa matruh"

    elif "sharm sheikh" in x:
        return "sharm sheikh"

    elif "monufia" in x or "guzal" in x:
        return "monufia"
    elif "sharqia" in x:
        return "sharqia"
    elif "fayoum" in x:
        return "fayoum"
    elif "ismailia" in x:
        return "ismailia"
    elif "qalyubia" in x:
        return "qalyubia"
    elif "suez" in x:
        return "suez"
    elif "beheira" in x:
        return "beheira"
    elif "kafr sheikh" in x:
        return "kafr sheikh"
    elif "aswan" in x:
        return "aswan"
    elif "sohag" in x:
        return "sohag"
    elif "minia" in x:
        return "minia"
    elif "asyut" in x:
        return "asyut"
    elif "qina" in x:
        return "qina"
    elif "beni suef" in x:
        return "beni suef"

    elif "sina" in x:
        return "sinai"

    elif (
        "new cairo" in x
        or "tagamoa" in x
        or "belagio" in x
        or "at nine" in x
        or "beta greens" in x
    ):
        return "new cairo"

    elif "cairo" in x:
        return "cairo"
    else:
        return x

df["City"] = df["Address"].apply(extract_city)
df["City"].value_counts()

City
new cairo               3140
madinaty                1162
other                    940
october                  708
new capital              546
cairo                    411
rehab                    305
sheikh zayed             277
heliopolis               212
shorouk                  192
maadi                    177
nasr city                175
hurghada                 151
alexandria               104
north coast               94
https aqarmap com eg      90
gharbia                   24
damietta                  19
naia west                 19
dakahlia                  15
marsa matruh              11
suez                      10
qalyubia                  10
novalist                  10
port said                  8
sharqia                    8
sohag                      8
ismailia                   7
monufia                    6
sharm sheikh               4
ras sidr                   4
qina                       4
aswan                      3
fayoum                     3
asyut    

In [34]:
df = df.drop(["Address"], axis=1)

### Dropping Price_Per_M2

`Dropping Price_Per_M2 column because nearly the half is missing or infinite value and it introduces multicolinearity`

In [35]:
df = df.drop(['Price_Per_M2'], axis=1)

In [36]:
df['Floor'].unique()

<ArrowStringArray>
[              nan,             '2.0',             '0.0',             '3.0',
            '11.0',             '7.0',             '1.0',             '4.0',
             '9.0',             '5.0',             '6.0',            '10.0',
            '13.0',            '12.0',            '18.0',             '8.0',
            '45.0',            '15.0',            '14.0',            '25.0',
            '17.0',            '16.0',   'Not Mentioned', 'Ground/Basement',
     'First Floor', 'Last Floor/Roof',   'Typical Floor']
Length: 27, dtype: str

### Floor column

`Floor column have overlapped values and Typical Floor and Last Floor/Roof holds a wide range of possible values, so to make data consistent we need need to categorize floors into bins`

In [37]:
df['Floor'] = df['Floor'].replace('Not Mentioned', np.nan)

In [38]:
floor_bins = {
    "First Floors/Ground": ["First Floor", "Ground/Basement", "0.0", "1.0", "2.0"],
    "Typical Floors": [
        "Typical Floor",
        "3.0",
        "4.0" ,"5.0",
        "6.0",
    ],
    "High Floors": ["7.0", "8.0" ,"9.0", "10.0", "11.0", "12.0", "13.0", "14.0", "15.0"],
    "Last Floors/Roof": ["16.0", "17.0", "18.0", "25.0", "45.0", "Last Floor/Roof"],
}

reverse_map = {val: category for category, values in floor_bins.items() for val in values}
df['Floor'] = df['Floor'].map(reverse_map)
df['Floor'].sample(50)

3629                    NaN
7827    First Floors/Ground
6954                    NaN
3079                    NaN
199             High Floors
1757    First Floors/Ground
3667                    NaN
1243    First Floors/Ground
578     First Floors/Ground
6305    First Floors/Ground
7615                    NaN
1969                    NaN
3015         Typical Floors
3947                    NaN
916     First Floors/Ground
216     First Floors/Ground
3071                    NaN
2180            High Floors
4972                    NaN
4292                    NaN
1897         Typical Floors
5756    First Floors/Ground
1444    First Floors/Ground
5559                    NaN
3417                    NaN
5572                    NaN
787     First Floors/Ground
881     First Floors/Ground
4485                    NaN
2934       Last Floors/Roof
2373         Typical Floors
2948            High Floors
8195    First Floors/Ground
6397                    NaN
7707                    NaN
4522                

`We need to fix the infinite value in the Price_Per_M2 by replacing it with the maximum non-infinite value`

### duplicates

`Handling duplicates`

In [39]:
df.astype(str).duplicated().sum()

np.int64(60)

In [40]:
df.astype(str).drop_duplicates(inplace=True)

In [41]:
df["Area_Per_Room"] =  df["Area"] / df["Rooms"]

df["Area_Per_Room"] = df["Area_Per_Room"].replace([np.inf, -np.inf], np.nan)

df["Rooms_per_Bathroom"] = df["Rooms"] / (df["Bathrooms"] + 1)
df["Rooms_per_Bathroom"] = df["Rooms_per_Bathroom"].replace([np.inf, -np.inf], np.nan)


df["Is_Large"] = (df["Area"] > df["Area"].quantile(0.75)).astype(int)
df["Is_Large"] = df["Is_Large"].replace([np.inf, -np.inf], np.nan)

### Data Splitting

`Splitting the data before preprocessing to prevent data leakage`

In [42]:
df = df.dropna(subset=["Price"])

In [43]:
df.info()

<class 'pandas.DataFrame'>
Index: 8864 entries, 0 to 8864
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Type                8864 non-null   str    
 1   Price               8864 non-null   float64
 2   Area                5791 non-null   float64
 3   Rooms               6918 non-null   float64
 4   Bathrooms           7017 non-null   float64
 5   Floor               4729 non-null   str    
 6   Finish              8864 non-null   int64  
 7   View                8838 non-null   str    
 8   Amenities_Count     8864 non-null   int64  
 9   Is_Installments     8864 non-null   int64  
 10  has_utilities       8864 non-null   int64  
 11  has_security        8864 non-null   int64  
 12  has_luxury          8864 non-null   int64  
 13  has_access          8864 non-null   int64  
 14  City                8864 non-null   str    
 15  Area_Per_Room       4821 non-null   float64
 16  Rooms_per_Bathroom  64

In [46]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('Price', axis=1), df['Price'], test_size=0.2, random_state=42)

### Data transformations

In [47]:
X_train.head()

,Type,Area,Rooms,Bathrooms,Floor,Finish,View,Amenities_Count,Is_Installments,has_utilities,has_security,has_luxury,has_access,City,Area_Per_Room,Rooms_per_Bathroom,Is_Large
1480,Villa,640.0,5.0,4.0,NaN,1,Garden,1,0,0,0,1,0,new cairo,128.0,1.00,1
745,Apartment,165.0,3.0,3.0,First Floors/Ground,1,Side Street,0,1,0,0,0,0,new cairo,55.0,0.75,0
6318,Office,82.0,2.0,1.0,NaN,1,Garden,1,0,0,0,1,0,madinaty,41.0,1.00,0
8805,Land,NaN,3.0,3.0,NaN,1,Garden,0,0,0,0,0,0,new cairo,NaN,0.75,0
4989,Apartment,NaN,3.0,2.0,First Floors/Ground,1,Pool,2,0,0,0,1,0,other,NaN,1.00,0


In [48]:
numeric_features = X_train.select_dtypes('number').columns
categorical_features = X_train.select_dtypes(exclude='number').columns

In [49]:
from sklearn.preprocessing import OrdinalEncoder
from category_encoders import TargetEncoder, BinaryEncoder

# --- Define column groups ---
ordinal_cols = {
    "Floor": [
        "First Floors/Ground",
        "Typical Floors",
        "High Floors",
        "Last Floors/Roof",
    ]
}
target_cols = ["City"]
binary_cols = ["Type", "View"]

for col, order in ordinal_cols.items():
    enc = OrdinalEncoder(
        categories=[order],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=np.nan,
    )
    X_train[[col]] = enc.fit_transform(X_train[[col]])
    X_test[[col]] = enc.transform(X_test[[col]])

target_enc = TargetEncoder(cols=target_cols, smoothing=10)
X_train[target_cols] = target_enc.fit_transform(X_train[target_cols], y_train)
X_test[target_cols] = target_enc.transform(X_test[target_cols])

for col in binary_cols:
    missing_train = X_train[col].isna()
    missing_test = X_test[col].isna()

    enc = BinaryEncoder(cols=[col], drop_invariant=True)
    X_train_enc = enc.fit_transform(X_train[[col]])
    X_test_enc = enc.transform(X_test[[col]])

    X_train_enc.loc[missing_train] = np.nan
    X_test_enc.loc[missing_test] = np.nan

    X_train = pd.concat([X_train.drop(col, axis=1), X_train_enc], axis=1)
    X_test = pd.concat([X_test.drop(col, axis=1), X_test_enc], axis=1)

X_train.head()

,Area,Rooms,Bathrooms,Floor,Finish,Amenities_Count,Is_Installments,has_utilities,has_security,has_luxury,...,Rooms_per_Bathroom,Is_Large,Type_0,Type_1,Type_2,Type_3,View_0,View_1,View_2,View_3
1480,640.0,5.0,4.0,-1.0,1,1,0,0,0,1,...,1.00,1,0,0,0,1,0.0,0.0,0.0,1.0
745,165.0,3.0,3.0,0.0,1,0,1,0,0,0,...,0.75,0,0,0,1,0,0.0,0.0,1.0,0.0
6318,82.0,2.0,1.0,-1.0,1,1,0,0,0,1,...,1.00,0,0,0,1,1,0.0,0.0,0.0,1.0
8805,NaN,3.0,3.0,-1.0,1,0,0,0,0,0,...,0.75,0,0,1,0,0,0.0,0.0,0.0,1.0
4989,NaN,3.0,2.0,0.0,1,2,0,0,0,1,...,1.00,0,0,0,1,0,0.0,0.0,1.0,1.0


In [53]:
X_train["City_x_Area"] = X_train["City"] * X_train["Area"]
X_test["City_x_Area"] = X_test["City"] * X_test["Area"]

In [54]:
# Select an estimator that automatically handels feature selections and overfitting
impute = impute = IterativeImputer(
    estimator=RandomForestRegressor(
        n_estimators=100,
        max_features=0.5,
        random_state=42,
        n_jobs=-1,
    ),
    max_iter=20, 
    tol=0.01, 
    initial_strategy="median",
    random_state=42,
)

X_train_processed = impute.fit_transform(X_train)
X_test_processed = impute.transform(X_test)
X_train_processed

c:\Users\amrta\.conda\envs\datascience\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


array([[6.40000000e+02, 5.00000000e+00, 4.00000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 6.50035021e+09],
       [1.65000000e+02, 3.00000000e+00, 3.00000000e+00, ...,
        1.00000000e+00, 0.00000000e+00, 1.67587154e+09],
       [8.20000000e+01, 2.00000000e+00, 1.00000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 6.00010800e+08],
       ...,
       [1.85000000e+02, 3.00000000e+00, 2.00000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 1.87900748e+09],
       [1.30000000e+02, 3.00000000e+00, 2.00000000e+00, ...,
        1.00000000e+00, 1.00000000e+00, 7.38025252e+08],
       [9.80000000e+01, 2.95857310e+00, 2.00000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 7.49861006e+08]], shape=(7091, 24))

`Avoiding scaling encoded features by selecting only numeric indecies to scale`

In [55]:
binary_flag_cols = [
    "has_utilities",
    "has_security",
    "has_luxury",
    "has_access",
    "Is_Installments",
]

binary_encoded_cols = [
    col
    for col in X_train.columns
    if any(col.startswith(base) for base in ["Type_", "View_"])
]

cols_to_exclude = binary_flag_cols + binary_encoded_cols

cols_to_scale = [
    col
    for col in X_train.columns
    if col not in cols_to_exclude and X_train[col].dtype in ["float64", "int64"]
]
cols_to_scale

['Area',
 'Rooms',
 'Bathrooms',
 'Floor',
 'Finish',
 'Amenities_Count',
 'City',
 'Area_Per_Room',
 'Rooms_per_Bathroom',
 'Is_Large',
 'City_x_Area']

In [56]:
scaler = StandardScaler()
X_train_processed_df = pd.DataFrame(
    X_train_processed, columns=X_train.columns, index=X_train.index
)
X_test_processed_df = pd.DataFrame(
    X_test_processed, columns=X_train.columns, index=X_test.index
)

X_train_processed_df[cols_to_scale] = scaler.fit_transform(
    X_train_processed_df[cols_to_scale]
)
X_test_processed_df[cols_to_scale] = scaler.transform(
    X_test_processed_df[cols_to_scale]
)

X_train_processed_df.head()

,Area,Rooms,Bathrooms,Floor,Finish,Amenities_Count,Is_Installments,has_utilities,has_security,has_luxury,...,Is_Large,Type_0,Type_1,Type_2,Type_3,View_0,View_1,View_2,View_3,City_x_Area
1480,5.421316,-0.044061,0.130728,-0.825127,0.228887,-0.232695,0.0,0.0,0.0,1.0,...,2.338780,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,6.017250
745,-0.027207,-0.044061,-0.006116,0.177096,0.228887,-0.860802,1.0,0.0,0.0,0.0,...,-0.427573,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.303479
6318,-0.979264,-0.044061,-0.279803,-0.825127,0.228887,-0.232695,0.0,0.0,0.0,1.0,...,-0.427573,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,-0.970694
8805,-0.150630,-0.044061,-0.006116,-0.825127,0.228887,-0.860802,0.0,0.0,0.0,0.0,...,-0.427573,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.171882
4989,0.028885,-0.044061,-0.142959,0.177096,0.228887,0.395413,0.0,0.0,0.0,1.0,...,-0.427573,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,-0.149113
